In [9]:
source ../modules/setup.tcl

set year 2017
set day 22

aoc::get-puzzle $year $day 1
# aoc::get-puzzle $year $day 1 2
set input [string trim [aoc::get-input $year $day]]
jupyter::html "<h2>Input</h2>"
jupyter::display "text/plain" [string range $input 0 100]...;

(cached)

--- Day 22: Sporifica Virus --- Diagnostics indicate that the local grid computing cluster has been contaminated with the Sporifica Virus . The grid computing cluster is a seemingly- infinite two-dimensional grid of compute nodes. Each node is either clean or infected by the virus. To prevent overloading the nodes (which would render them useless to the virus) or detection by system administrators, exactly one virus carrier moves through the network, infecting or cleaning nodes as it moves. The virus carrier is always located on a single node in the network (the current node ) and keeps track of the direction it is facing. To avoid detection, the virus carrier works in bursts; in each burst, it wakes up , does some work , and goes back to sleep . The following steps are all executed in order one time each burst: 
 If the current node is infected , it turns to its right . Otherwise, it turns to its left . (Turning is done in-place; the current node does not change.) If the current node is clean , it becomes infected . Otherwise, it becomes cleaned . (This is done after the node is considered for the purposes of changing direction.) The virus carrier moves forward one node in the direction it is facing. 
 Diagnostics have also provided a map of the node infection status (your puzzle input). Clean nodes are shown as . ; infected nodes are shown as # . This map only shows the center of the grid; there are many more nodes beyond those shown, but none of them are currently infected. The virus carrier begins in the middle of the map facing up . For example, suppose you are given a map like this: ..#
#..
...
 Then, the middle of the infinite grid looks like this, with the virus carrier's position marked with [ ] : . . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . # . . .
. . . #[.]. . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
 The virus carrier is on a clean node, so it turns left , infects the node, and moves left: . . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . # . . .
. . .[#]# . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
 The virus carrier is on an infected node, so it turns right , cleans the node, and moves up: . . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . .[.]. # . . .
. . . . # . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
 Four times in a row, the virus carrier finds a clean , infects it, turns left , and moves forward, ending in the same place and still facing up: . . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . #[#]. # . . .
. . # # # . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
 Now on the same node as before, it sees an infection, which causes it to turn right , clean the node, and move forward: . . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . # .[.]# . . .
. . # # # . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
 After the above actions, a total of 7 bursts of activity had taken place. Of them, 5 bursts of activity caused an infection. After a total of 70 , the grid looks like this, with the virus carrier facing up: . . . . . # # . .
. . . . # . . # .
. . . # . . . . #
. . # . #[.]. . #
. . # . # . . # .
. . . . . # # . .
. . . . . . . . .
. . . . . . . . .
 By this time, 41 bursts of activity caused an infection (though most of those nodes have since been cleaned). After a total of 10000 bursts of activity, 5587 bursts will have caused an infection. Given your actual map, after 10000 bursts of activity, how many bursts cause a node to become infected ? (Do not count nodes that begin infected.)

(cached)

Input

.#...#.#.##..##....##.#.#
###.###..##...##.##....##
....#.###..#...#####..#.#
.##.######..###.##..#.....

In [2]:
proc init {input stateVar} {
    upvar $stateVar state
    set state(infections) 0 
    set state(bursts) 0 
    set state(dx) 0
    set state(dy) -1
    set rows [split $input \n]
    set start [expr {[llength $rows]/2}]
    set state(x) $start
    set state(y) $start
    set y 0
    foreach row $rows {
        set x 0
        foreach cell [split $row ""] {
            if {$cell eq "#"} {
                set state($x,$y) $cell
            }
            incr x
        }
        incr y

    }
}

proc printgrid {stateVar} {
    upvar $stateVar state
    set pois [lmap x [array names state {*,*}] {split $x ,}]
    lappend pois [list $state(x) $state(y)]
#     puts $pois
    set byx [lsort -integer -index 0 $pois]
    set minx [lindex [lindex $byx 0] 0]
    set maxx [lindex [lindex $byx end] 0]
    set byy [lsort -integer -index 1 $pois]
    set miny [lindex [lindex $byy 0] 1]
    set maxy [lindex [lindex $byy end] 1]
    incr minx -1
    incr maxx
    incr miny -1
    incr maxy
    puts =======
#     puts $minx:$maxx,$miny:$maxy
    set rows {}
    set posx $state(x)
    set posy $state(y)
    for {set y $miny} {$y<=$maxy} {incr y} {
        set cols {}
        for {set x $minx} {$x<=$maxx} {incr x} {
#             puts "$x,$y"
            if {$x == $posx && $y == $posy} {
                lappend cols @
            } elseif {[info exists state($x,$y)]} {
               lappend cols $state($x,$y)
            } {
               lappend cols .
            }
                
        }
        lappend rows $cols
    }

#     set posrow [lindex $rows $posy+1]
# #     puts "from:\t $posrow"
#     set sx [expr {($posx+1) * 2}]
    
#     set newrow [string replace $posrow $sx-1 $x-1 {[}]
#     set newrow [string replace $newrow $sx+1 $sx+1 {]}]
#      puts "to:\t $newrow"
#     lset rows $posy+1 $newrow
#      puts $newrow
    puts [join $rows \n] 

        puts "Infections\t$state(infections)"
        puts "Pos:\t\t$state(x):$state(y)"
        puts "Dir:\t\t$state(dx):$state(dy)"
    
}
namespace import tcl::mathop::*

proc step1 {stateVar} {
    upvar $stateVar state
    set dx $state(dx)
    set dy $state(dy)
    set x $state(x)
    set y $state(y)
    if {[info exists state($x,$y)]} {
        set state(dx) [- $dy]
        set state(dy) $dx
        unset state($x,$y)
    } {
        set state($x,$y) "#"
        incr state(infections)
        set state(dx) $dy
        set state(dy) [- $dx]
    }
    incr state(x) $state(dx)
    incr state(y) $state(dy)
    incr state(bursts)
    
}

In [6]:
set testinput {..#
#..
...}
unset -nocomplain state
init $testinput state
# parray state

# puts xxxxxxxxxx
# step1 state

# parray state

# puts xxxxxxxxxx
# step1 state

# parray state

puts [time { time {
#  printgrid state
#  parray state
    step1 state
} 7 }]
puts $state(infections)


30 microseconds per iteration
5


In [7]:
proc step2 {stateVar} {
    upvar $stateVar state
    set dx $state(dx)
    set dy $state(dy)
    set x $state(x)
    set y $state(y)
    incr state(bursts)
    set cell .
    if {[info exists state($x,$y)]} {
       set cell $state($x,$y) 
    }
    switch -exact $cell {
        . {
            set state($x,$y) W
            set state(dx) $dy
            set state(dy) [- $dx]
        }
        W {
            set state($x,$y) "#"
            incr state(infections)
        }
        "#" {
            set state($x,$y) "F"
            dict set grid [list $x $y] "F"
            set state(dx) [- $dy]
            set state(dy) $dx
        }
        "F" {
            unset state($x,$y) 
            set state(dx) [- $dx]
            set state(dy) [- $dy]
        }

    }
    incr state(x) $state(dx)
    incr state(y) $state(dy)
}

In [8]:
aoc::parts {
    set input [yield]
    unset -nocomplain state
    init $input state
    time { step1 state } 10000
    yield $state(infections)

    unset -nocomplain state
    init $input state
    time { step2 state } 10000000
    yield $state(infections)
}
aoc::results 

Part1	5322 (22466 us)
Part2	2512079 (11079141 us)


5322 2512079